In [1]:
#1-Defining the class which logs me in to the viewer website with more flexibility than fct used in other code
#CandidateViewerQuery and CandidateViewerRegistrar
import requests
from pprint import pprint

class AutomationAPI:
    def __init__(self, base_url, survey_id, username, password):
        self.url = base_url
        self.survey_id = survey_id
        self.username = username
        self.password = password
        self.survey_aval = []
        self.session = requests.Session()

        # Login
        self._call("login", username=username, password=password)
        
        # Register endpoints
        self._register_endpoints()
        
        # Get list of surveys
        self.survey_aval = self._call("get_all_surveys")["surveys"]

        # Check if survey_id is valid
        if survey_id not in self.survey_aval:
            raise ValueError(f"Survey ID {survey_id} is not available. Available surveys: ", self.survey_aval)
        
        # Set survey
        self._call("set_survey", survey=survey_id)

    def _call(self, endpoint, **params):
        response = self.session.post(
            self.url,
            params={"endpoint": endpoint},  # GET
            data=params                      # POST
        )
        response.raise_for_status()
        
        try:
            data = response.json()
            if data["status"] != "success":
                raise RuntimeError(data["message"])
        except ValueError:
            raise RuntimeError("Invalid JSON response", response.text)
        
        return data["data"]

    def _register_endpoints(self):
        for endpoint in self._call("get_endpoints_aval")["endpoints"]:
            name = endpoint["name"]
            param_names = endpoint["params"]

            if hasattr(self, name):
                continue

            def make_method(endpoint_name, endpoint_params):
                def method(self, **kwargs):
                    missing = [p for p in endpoint_params if p not in kwargs]
                    if missing:
                        raise ValueError(f"Missing parameter(s): {', '.join(missing)}")
                    return self._call(endpoint_name, **kwargs)
                method.__name__ = endpoint_name
                method.__doc__  = f"Params: {', '.join(endpoint_params)}"
                return method

            setattr(self.__class__, name, make_method(name, param_names))

In [2]:
#2-Calling the function for the test folder
api = AutomationAPI(
    "https://sps.chimenet.ca/candidates/index.php?automation", "multiday", "Viewer Bot", "v4A13BNYwqU5okUZE^h9c&x*blzHrYMi"
)

In [7]:
#3-Function to get file info from specific rating(wenke's one do not work for "Possibly intermittent"

files = api.get_files(folder="single_day_1")
count = len(files["files"])
#print(count)
#print(files)
#for files in files["files"]:
   # print(files["checked"]["tags"])

print(api.get_file_details(folder="test_21", file="Multi_Pointing_Groups_f_0.876_DM_192.175_698552b2dad56d5360166ff6&folder"))

{'status': False}


In [3]:
selected = []

for f in files["files"]:
    tags = f["checked"]["tags"]
    
    if any(t["tag"] == "Possibly intermittent" for t in tags):
        selected.append(f)

NameError: name 'files' is not defined

In [8]:
import sps_databases
import subprocess
from cfbm.bm_data import get_data
import os
import numpy as np
from sps_databases import db_utils, db_api
import scipy
from datetime import datetime, timedelta
from scheduler.run_as_service import run_as_service
import pickle

#Step_1-Query website candidates and put them in a list
from sps_pipeline.candidate_viewer import CandidateViewerQuery
from sps_pipeline.candidate_viewer import CandidateViewerRegistrar
from multiday_search import multidayfold_pipeline
import os

# Database configuration
db_config = {
    'host': 'sps-archiver1',
    'user': 'automation',
    'port': 3306,
    'password': '',#no password for automation user
    'database': 'champss'
}


all_candidates = []
with CandidateViewerQuery(survey='multiday', db_config=db_config,) as query:
    
    folder = "single_day_1"  
    print(f"\nProcessing folder: {folder}")
    
    candidates = query.get_ratings(
        folder=folder,
        with_metadata=True
    )
    print(

SyntaxError: incomplete input (290642400.py, line 38)